# Lab: Hybrid RAG (Vector + Graph)
In this notebook, we bridge the gap between **Semantic Vector Search** and **Structural Graph Traversal**. 

Standard Vector Search finds documents by overall meaning, while Knowledge Graphs excel at multi-hop relationship tracing. By combining them, we use vectors to find the most relevant "seed nodes" in Neo4j, and then use Cypher queries to expand those seeds into rich, multi-hop relationship paths for our LLM.

In [42]:
!pip install -qU pypdf neo4j sentence-transformers langchain-huggingface langchain-openai yfiles-jupyter-graphs-for-neo4j


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 1: Imports

In [43]:
import os
import json
import requests
from pypdf import PdfReader
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate

In [ ]:
# Neo4j Database Credentials
NEO4J_URI = "YOUR-NEO4J_URI"
NEO4J_USER = "YOUR-NEO4J_USER"
NEO4J_PASSWORD = "YOUR-NEO4J_PASSWORD"

# Open persistent database connection
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Success: Connected to Neo4j Database!")

Success: Connected to Neo4j Database!


### Step 2: Configure the LLM

In [ ]:
llm = ChatOpenAI(
    openai_api_key="your-api-key",
    openai_api_base="https://openrouter.ai/api/v1",
    model_name="nvidia/nemotron-3-ultra-550b-a55b:free",
    temperature=0.0
)
print("LangChain LLM configured successfully!")

LangChain LLM configured successfully!


### Step 3: Download PDF from Link & Extract Text

In [46]:
os.makedirs("data", exist_ok=True)

PDF_URL = "https://arxiv.org/pdf/1706.03762.pdf"
PDF_FILENAME = "data/downloaded_paper.pdf"

print(f"Downloading PDF from link: {PDF_URL}...")
response = requests.get(PDF_URL)
response.raise_for_status()

with open(PDF_FILENAME, "wb") as f:
    f.write(response.content)
print(f"PDF downloaded successfully to {PDF_FILENAME}!")

print("Extracting text from PDF...")
# Reading the text from the PDF
reader = PdfReader(PDF_FILENAME)
document_text = ""

for page in reader.pages:
    document_text += page.extract_text() + "\n"
    # Stop after 1500 characters, so the demo runs fast
    if len(document_text) >= 1500:
        document_text = document_text[:1500]
        break

print(f"Extracted {len(document_text)} characters of text.")

PDF downloaded successfully to data/downloaded_paper.pdf!
Extracting text from PDF...
Extracted 1500 characters of text.


### Step 4: Extract Graph Structure 

In [47]:
print("Asking LLM to extract graph nodes and edges...")

prompt = f"""
You are a data extraction AI. Read the text and extract a knowledge graph.
Identify key concepts and the relationships between them.

TEXT:
{document_text}

CRITICAL INSTRUCTION: Reply ONLY with valid JSON.
Format exactly like this:
{{
  "nodes": [ {{"name": "Concept 1"}}, {{"name": "Concept 2"}} ],
  "relationships": [ {{"source": "Concept 1", "target": "Concept 2", "type": "RELATES_TO"}} ]
}}
"""

response = llm.invoke(prompt)

raw_output = response.content.strip()
# The AI sometimes adds ```json marks around its answer. We remove them here.
if raw_output.startswith("```json"):
    raw_output = raw_output[7:-3].strip()
elif raw_output.startswith("```"):
    raw_output = raw_output[3:-3].strip()

graph_data = json.loads(raw_output)
print(f"Extracted {len(graph_data.get('nodes', []))} nodes and {len(graph_data.get('relationships', []))} relationships!")

Asking LLM to extract graph nodes and edges...
Extracted 18 nodes and 23 relationships!


### Step 5: Save Graph to Neo4j (Raw Cypher)

In [48]:
# Save nodes to Neo4j
node_query = """
UNWIND $nodes AS node
MERGE (c:Concept {name: node.name})
"""

with driver.session() as session:
    session.run(node_query, nodes=graph_data.get("nodes", []))

print("Nodes saved to Neo4j!")

Nodes saved to Neo4j!


In [49]:
# Save relationships to Neo4j
rel_query = """
UNWIND $relationships AS rel
MATCH (source:Concept {name: rel.source})
MATCH (target:Concept {name: rel.target})
CALL apoc.create.relationship(source, replace(toUpper(rel.type), ' ', '_'), {}, target) YIELD rel AS r
RETURN count(r)
"""
# This special command lets us name the relationship using a variable (like "BASED_ON").
# Normal Cypher code cannot do this.

with driver.session() as session:
    if graph_data.get("relationships"):
        session.run(rel_query, relationships=graph_data.get("relationships", []))

print("Relationships saved to Neo4j!")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description="warn: feature deprecated with replacement. apoc.create.relationship is deprecated. It is replaced by Cypher's dynamic types: `CREATE (from)-[n:$(relType)]->(to) SET n = props`.", position=<SummaryInputPosition line=5, column=1, offset=114>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 114, 'line': 5, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nUNWIND $relationships AS rel\nMATCH (source:Concept {name: rel.source})\nMATCH (target:Concept {name: rel.target})\nCALL apoc.create.relationship(source, replace(toUpper(rel.type), ' ', '_'), {}, target) YIELD rel AS r\nRETURN count(r)\n"


Relationships saved to Neo4j!


### Step 6: Initialize Embedding Model & Create Vector Index

In [50]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# This model always makes vectors of size 384. The number below must match.
index_query = """
CREATE VECTOR INDEX `concept_embeddings` IF NOT EXISTS 
FOR (c:Concept) ON (c.embedding) 
OPTIONS {
    indexConfig: { 
        `vector.dimensions`: 384, 
        `vector.similarity_function`: 'cosine' 
    }
}
"""

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Step 7: Generate and Store Vector Embeddings

In [51]:
with driver.session() as session:
    # Only get nodes that do not have a vector yet, so we don't do the work twice
    result = session.run("MATCH (c:Concept) WHERE c.embedding IS NULL RETURN elementId(c) as node_id, c.name AS text")
    records = [record for record in result]
    
    if not records:
        print("All nodes already have embeddings.")
    else:
        print(f"Generating vectors for {len(records)} nodes...")
        for record in records:
            vector = embeddings.embed_query(record["text"])
            session.run(
                "MATCH (c) WHERE elementId(c) = $node_id SET c.embedding = $vector", 
                node_id=record["node_id"], 
                vector=vector
            )
        print("Success: All graph concepts vectorized!")

Generating vectors for 18 nodes...
Success: All graph concepts vectorized!


### Step 8: Define the Hybrid Retrieval Function

In [ ]:
def execute_hybrid_retrieval(driver, question, top_k=3):
    print(f"User Question: '{question}'\n")

    # Turn the question into a vector, using the same method used for the graph nodes
    question_vector = embeddings.embed_query(question)

    # First find the nodes closest in meaning to the question.
    # Then look at what each of those nodes is connected to.
    hybrid_query = """
    CALL db.index.vector.queryNodes('concept_embeddings', $top_k, $question_vector)
    YIELD node AS seed, score
    MATCH (seed)-[r]-(neighbor:Concept)
    RETURN seed.name AS Seed_Node, score AS Semantic_Score, type(r) AS Relationship, neighbor.name AS Connected_Node
    """

    retrieved_facts = []
    with driver.session() as session:
        result = session.run(hybrid_query, top_k=top_k, question_vector=question_vector)

        print("RETRIEVED HYBRID PATHS:")
        print("-" * 60)
        for record in result:
            # Turn each result into one easy-to-read line of text
            fact = f"(Similarity: {record['Semantic_Score']:.2f}) {record['Seed_Node']} --[{record['Relationship']}]--> {record['Connected_Node']}"
            retrieved_facts.append(fact)
            print(fact)

    return retrieved_facts

### Step 9: Generate Answer using LangChain Prompt Pipelines

In [53]:
def generate_hybrid_answer(question, retrieved_facts):
    if not retrieved_facts:
        return "No relevant context found in the database."
        
    facts_block = "\n".join([f"- {f}" for f in retrieved_facts])
    
    template = """
    You are an expert AI research assistant using a Neo4j Knowledge Graph.
    Answer the question using ONLY the connected relationship paths provided below.

    Graph Relationships:
    {facts_block}

    Question: {question}

    Output your response in EXACTLY two sections:
    --- FINAL ANSWER ---
    [Provide a direct, simple, 1-sentence answer.]

    --- AI TRACING & EXPLAINABILITY ---
    [Explain step-by-step how the answer was derived from the Neo4j graph context.]
    """
    
    prompt = PromptTemplate(template=template, input_variables=["facts_block", "question"])
    # The | joins the steps together: first fill in the prompt, then send it to the AI
    chain = prompt | llm
    
    response = chain.invoke({"facts_block": facts_block, "question": question})
    return response.content

### Step 10: Querying

In [ ]:
query = "What models or mechanisms are discussed in the document?"

facts = execute_hybrid_retrieval(driver, query, top_k=3)

print(generate_hybrid_answer(query, facts))

User Question: 'What models or mechanisms are discussed in the document?'



Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=5, offset=5>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 5, 'line': 2, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    CALL db.index.vector.queryNodes('concept_embeddings', $top_k, $question_vector)\n    YIELD node AS seed, score\n    MATCH (seed)-[r]-(neighbor:Concept)\n    RETURN seed.name AS Seed_Node, score AS Semantic_Score, type(r) AS Relationship, neighbor.name AS Connected_Node\n    "


RETRIEVED HYBRID PATHS:
------------------------------------------------------------
(Similarity: 0.68) Sequence Transduction Models --[BASED_ON]--> Recurrent Neural Networks
(Similarity: 0.68) Sequence Transduction Models --[BASED_ON]--> Convolutional Neural Networks
(Similarity: 0.68) Sequence Transduction Models --[INCLUDES]--> Encoder
(Similarity: 0.68) Sequence Transduction Models --[INCLUDES]--> Decoder
(Similarity: 0.65) Google Research --[AUTHORS_AFFILIATED_WITH]--> Attention Is All You Need
--- FINAL ANSWER ---
The document discusses Sequence Transduction Models (including Encoder and Decoder components), Recurrent Neural Networks, Convolutional Neural Networks, and the Attention Is All You Need mechanism.

--- AI TRACING & EXPLAINABILITY ---
Step 1: Identify the central document node from the graph - "Attention Is All You Need" is connected to "Google Research" via AUTHORS_AFFILIATED_WITH relationship.
Step 2: Find all model/mechanism nodes connected to the document context -

In [55]:
from yfiles_jupyter_graphs_for_neo4j import Neo4jGraphWidget

widget = Neo4jGraphWidget(driver)
widget.show_cypher("MATCH (n:Concept)-[r]-(m:Concept) RETURN n, r, m")

GraphWidget(layout=Layout(height='680px', width='100%'))